# Reading the csv file and cropping the file to delete any records before 00:00 1 Jan 2010

In [1]:
# importing the pandas pacakge
import pandas as pd

# reading the csv file using pandas package and storing it in a variable, sop
# and structuring the date column using parse_dates
sop = pd.read_csv('air-quality-data-continuous.csv', parse_dates=['Date Time'], delimiter=';')

# filtering out the unrequired dates (dates before Jan 1, 2010 00:00)
sop = sop[(pd.to_datetime("2010-01-01 00:00:00+00:00") <= sop['Date Time'])]

sop.shape

# naming and storing the newly created csv file
sop.to_csv('crop.csv', sep= ';', index = False)

C:\Users\friya\AppData\Local\Temp\ipykernel_27896\2571647997.py:6: DtypeWarning: Columns (20) have mixed types. Specify dtype option on import or set low_memory=False.
  sop = pd.read_csv('air-quality-data-continuous.csv', parse_dates=['Date Time'], delimiter=';')


#  Filtering and removing any dud records where there is no value for SiteID or where there is a mismatch between SiteID and Location.

In [2]:
# importing the pandas pacakge with an alias
import pandas as pd

# creating a dictionary to store the stations
diction = {188: 'AURN Bristol Centre',
           203: 'Brislington Depot',
           206: 'Rupert Street',
           209: 'IKEA M32',
           213: 'Old Market',
           215: 'Parson Street School',
           228: 'Temple Meads Station',
           270: 'Wells Road',
           271: 'Trailer Portway P&R',
           375: 'Newfoundland Road Police Station',
           395: "Shiner's Garage",
           452: 'AURN St Pauls',
           447: 'Bath Road',
           459: 'Cheltenham Road \ Station Road',
           463: 'Fishponds Road',
           481: 'CREATE Centre Roof',
           500: 'Temple Way',
           501: 'Colston Avenue',
           672: 'Marlborough Street'
           }

#using pandas to read the crop.csv file
sopuru = pd.read_csv('crop.csv', delimiter = ';')

#creating empty dictionaries to place required and unrequired values respectively
present = set()
absent = set()

# looping through each row of the dataframe
for index, row in sopuru.iterrows(): 
    if row['SiteID'] in diction:
        present.add(row['SiteID']) # populating the 'present' dictionary with values found in 'diction' dictionary
        if diction[row['SiteID']] != row['Location']:
            sopuru.drop(row['SiteID'], inplace=True)
    else:
        absent.add(row["SiteID"]) # populating the 'absent' dictionary with values not found in 'diction' dictionary
        
# remove rows that have siteID in absent set
sopuru1 = sopuru[sopuru['SiteID'].isin(list(absent)) == False]

# previewing the final data
print(sopuru1.shape)

# writing the data into csv format
sopuru1.to_csv('clean.csv', index = False)

C:\Users\friya\AppData\Local\Temp\ipykernel_27896\2053831611.py:27: DtypeWarning: Columns (20) have mixed types. Specify dtype option on import or set low_memory=False.
  sopuru = pd.read_csv('crop.csv', delimiter = ';')


(862267, 23)


# Creating an SQL database and populating it with the tables and values from the cleaned CSV file

In [4]:
import pandas as pd
import numpy as np
import pymysql as pm

# Load data
print("Loading data...")
df = pd.read_csv('clean.csv')

# Take a reproducible sample of 86000 rows (because the original file size takes a very long time to execute)
print("Sampling data...")
df = df.sample(n=86000, random_state=42)

# Replace np.nan with None for MySQL compatibility (MySQL does not recognize null or nan values)
df = df.replace({np.nan: None})

# Connect to MySQL
print("Connecting to MySQL...")
conn = pm.connect(
    host='localhost',
    user='root',
    password=''  # Add your password if you have one
)
cursor = conn.cursor()

# Create new database
print("Creating database and tables...")
cursor.execute("DROP DATABASE IF EXISTS pollution_db2")
cursor.execute("CREATE DATABASE pollution_db2")
cursor.execute("USE pollution_db2")

# Create tables
cursor.execute("""
    CREATE TABLE Sites (
        SiteID INT PRIMARY KEY,
        Location VARCHAR(255),
        geo_point_2d VARCHAR(100)
    )
""")

cursor.execute("""
    CREATE TABLE Instrument (
        InstrumentID INT AUTO_INCREMENT PRIMARY KEY,
        SiteID INT,
        InstrumentType VARCHAR(255),
        DateStart DATETIME,
        DateEnd DATETIME,
        Current BOOLEAN,
        FOREIGN KEY (SiteID) REFERENCES Sites(SiteID)
    )
""")

cursor.execute("""
    CREATE TABLE Measurements (
        MeasurementID INT AUTO_INCREMENT PRIMARY KEY,
        DateTime DATETIME,
        SiteID INT,
        NOx FLOAT,
        NO2 FLOAT,
        NO FLOAT,
        PM10 FLOAT,
        NVPM10 FLOAT,
        VPM10 FLOAT,
        PM2_5 FLOAT,
        NVPM2_5 FLOAT,
        VPM2_5 FLOAT,
        CO FLOAT,
        O3 FLOAT,
        SO2 FLOAT,
        Temperature FLOAT,
        RH FLOAT,
        AirPressure FLOAT,
        FOREIGN KEY (SiteID) REFERENCES Sites(SiteID)
    )
""")

# Insert into Sites (Batch insert)
sites = df[['SiteID', 'Location', 'geo_point_2d']].drop_duplicates(subset='SiteID')
site_records = [tuple(row) for row in sites.values]
print(f"Inserting {len(site_records)} site records...")
cursor.executemany("""
    INSERT INTO Sites (SiteID, Location, geo_point_2d)
    VALUES (%s, %s, %s)
""", site_records)

# Insert into Instrument (Batch insert)
instruments = df[['SiteID', 'Instrument Type', 'DateStart', 'DateEnd', 'Current']].drop_duplicates()
instrument_records = [tuple(row) for row in instruments.values]
print(f"Inserting {len(instrument_records)} instrument records...")
cursor.executemany("""
    INSERT INTO Instrument (SiteID, InstrumentType, DateStart, DateEnd, Current)
    VALUES (%s, %s, %s, %s, %s)
""", instrument_records)

# Insert into Measurements (Batch insert)
measurement_columns = [
    'Date Time', 'SiteID', 'NOx', 'NO2', 'NO', 'PM10', 'NVPM10', 'VPM10',
    'PM2.5', 'NVPM2.5', 'VPM2.5', 'CO', 'O3', 'SO2', 'Temperature', 'RH', 'Air Pressure'
]

# Ensure the column names match table columns
measurements = df[measurement_columns]
# Rename columns to match SQL field names
measurements = measurements.rename(columns={
    'Date Time': 'DateTime',
    'PM2.5': 'PM2_5',
    'NVPM2.5': 'NVPM2_5',
    'VPM2.5': 'VPM2_5',
    'Air Pressure': 'AirPressure'
})

measurement_records = [tuple(row) for row in measurements.values]
print(f"Inserting {len(measurement_records)} measurement records...")
cursor.executemany("""
    INSERT INTO Measurements (
        DateTime, SiteID, NOx, NO2, NO, PM10, NVPM10, VPM10, PM2_5, NVPM2_5, VPM2_5,
        CO, O3, SO2, Temperature, RH, AirPressure
    ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
""", measurement_records)

# Finalize
conn.commit()
cursor.close()
conn.close()

print("Database population complete!")


Loading data...


C:\Users\friya\AppData\Local\Temp\ipykernel_27896\2296368081.py:7: DtypeWarning: Columns (20) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('clean.csv')


Sampling data...
Connecting to MySQL...
Creating database and tables...
Inserting 15 site records...
Inserting 15 instrument records...
Inserting 86000 measurement records...
Database population complete!


# Executing a sample SQL query on the first 100 rows of data created

In [5]:
#df = pd.read_csv('clean.csv')
subset = df.head(100)

with open('insert-100.sql', 'w') as f:
    for _, row in subset.iterrows():
        insert_stmt = f"""
INSERT INTO Measurements (
    DateTime, SiteID, NOx, NO2, NO, PM10, NVPM10, VPM10, PM2_5, NVPM2_5, VPM2_5,
    CO, O3, SO2, Temperature, RH, AirPressure
) VALUES (
    '{row['Date Time']}', {row.SiteID}, {row.NOx}, {row.NO2}, {row.NO},
    {row.PM10}, {row.NVPM10}, {row.VPM10}, {row['PM2.5']}, {row['NVPM2.5']},
    {row['VPM2.5']}, {row.CO}, {row.O3}, {row.SO2}, {row.Temperature},
    {row.RH}, {row['Air Pressure']}
);
"""
        f.write(insert_stmt)


# Executing even more SQL queries...

# Return the date/time, station name and the highest recorded value of nitrogen oxide (NOx) found in the dataset for the year 2022.

# Return the mean values of PM2.5 (particulate matter <2.5 micron diameter) & VPM2.5 (volatile particulate matter <2.5 micron diameter) by each station for the year 2019 for readings taken at 08:00 hours (peak traffic intensity).

# Return the mean values of PM2.5 (particulate matter <2.5 micron diameter) & VPM2.5 (volatile particulate matter <2.5 micron diameter) by each station taken at 08:00 hours (peak traffic intensity) from year 2010 to 2022.

In [11]:
import pandas as pd
import pymysql as pm

# Connect to the MySQL database
conn = pm.connect(
    host='localhost',
    user='root',
    password='',
    database='pollution_db2',
    cursorclass=pm.cursors.DictCursor  # <- KEY: Make results return as dictionaries
)

cursor = conn.cursor()

# Define the SQL test queries
queries = {
    "highest_nox_2022": """ 
        SELECT 
            m.DateTime,
            s.Location AS StationName,
            m.NOx
        FROM 
            Measurements m
        JOIN 
            Sites s ON m.SiteID = s.SiteID
        WHERE 
            YEAR(m.DateTime) = 2022
            AND m.NOx IS NOT NULL
        ORDER BY 
            m.NOx DESC
        LIMIT 1;
    """,
    "mean_pm2_5_vpm2_5_2019_8am": """
        SELECT 
            s.Location AS StationName,
            AVG(m.PM2_5) AS Avg_PM2_5,
            AVG(m.VPM2_5) AS Avg_VPM2_5
        FROM 
            Measurements m
        JOIN 
            Sites s ON m.SiteID = s.SiteID
        WHERE 
            YEAR(m.DateTime) = 2019
            AND HOUR(m.DateTime) = 8
        GROUP BY 
            s.Location;
    """,
    "mean_pm2_5_vpm2_5_2010_2022_8am": """
        SELECT 
            YEAR(m.DateTime) AS Year,
            s.Location AS StationName,
            AVG(m.PM2_5) AS Avg_PM2_5,
            AVG(m.VPM2_5) AS Avg_VPM2_5
        FROM 
            Measurements m
        JOIN 
            Sites s ON m.SiteID = s.SiteID
        WHERE 
            YEAR(m.DateTime) BETWEEN 2010 AND 2022
            AND HOUR(m.DateTime) = 8
        GROUP BY 
            YEAR(m.DateTime), s.Location
        ORDER BY 
            Year, StationName;
    """
}

# Execute and store results
results = {}
for key, query in queries.items():
    cursor.execute(query)
    results[key] = cursor.fetchall()  # Now each result is a list of dictionaries

# Close the connection
cursor.close()
conn.close()

# Convert results to Pandas DataFrames
dfs = {key: pd.DataFrame(value) for key, value in results.items()}

'''# Example display
import ace_tools as tools
tools.display_dataframe_to_user(name="Highest NOx in 2022", dataframe=dfs["highest_nox_2022"])'''

# Display the Highest NOx recorded in 2022
print("=== Highest NOx recorded in 2022 ===")
print(dfs["highest_nox_2022"])
print("\n")  # Add a line break for neatness

# Display the Mean PM2.5 and VPM2.5 values by station for 2019 at 08:00
print("=== Mean PM2.5 and VPM2.5 values by station (2019, 08:00 hours) ===")
print(dfs["mean_pm2_5_vpm2_5_2019_8am"])
print("\n")

# Display the Mean PM2.5 and VPM2.5 values by station for 2010-2022 at 08:00
print("=== Mean PM2.5 and VPM2.5 values by station (2010–2022, 08:00 hours) ===")
print(dfs["mean_pm2_5_vpm2_5_2010_2022_8am"])

=== Highest NOx recorded in 2022 ===
             DateTime     StationName     NOx
0 2022-01-28 08:00:00  Colston Avenue  942.25


=== Mean PM2.5 and VPM2.5 values by station (2019, 08:00 hours) ===
            StationName  Avg_PM2_5 Avg_VPM2_5
0         AURN St Pauls  10.071429       None
1     Brislington Depot        NaN       None
2        Colston Avenue        NaN       None
3        Fishponds Road        NaN       None
4  Parson Street School  11.780600       None
5            Temple Way        NaN       None
6            Wells Road        NaN       None


=== Mean PM2.5 and VPM2.5 values by station (2010–2022, 08:00 hours) ===
     Year                     StationName  Avg_PM2_5  Avg_VPM2_5
0    2010                   AURN St Pauls  15.217391    3.130435
1    2010                       Bath Road        NaN         NaN
2    2010               Brislington Depot        NaN         NaN
3    2010  Cheltenham Road \ Station Road        NaN         NaN
4    2010                  Fishpo

# Model and implement selected NoSQL databases for a sample row of data: Site ID = 501.

## 1. Key-Value Model (e.g., Redis, DynamoDB)

In [13]:
import json
import os

# Filter data for SiteID = 501
station_df = df[df['SiteID'] == 501].copy()
station_df.reset_index(drop=True, inplace=True)

# Format data as key-value store, key = Site:501:<timestamp>, value = JSON string of measurements
kv_store = {}

for _, row in station_df.iterrows():
    key = f"Station:501:{row['Date Time']}"
    value = {
        "NOx": row.NOx,
        "NO2": row.NO2,
        "NO": row.NO,
        "PM10": row.PM10,
        "NVPM10": row.NVPM10,
        "VPM10": row.VPM10,
        "PM2.5": row["PM2.5"],
        "NVPM2.5": row["NVPM2.5"],
        "VPM2.5": row["VPM2.5"],
        "CO": row.CO,
        "O3": row.O3,
        "SO2": row.SO2,
        "Temperature": row.Temperature,
        "RH": row.RH,
        "AirPressure": row["Air Pressure"]
    }
    kv_store[key] = value

# Save as JSON file
output_dir = 'json_outputs'
os.makedirs(output_dir, exist_ok=True)

kv_output_path = os.path.join(output_dir, 'kv_store_501.json')
with open(kv_output_path, 'w') as f:
    json.dump(kv_store, f, indent=2)

print(f"JSON file successfully saved at {kv_output_path}")


JSON file successfully saved at json_outputs\kv_store_501.json


## 2. XML Document Model (e.g., BaseX, eXist-db)

In [16]:
import os
import pandas as pd
import xml.etree.ElementTree as ET
from xml.etree.ElementTree import Element, SubElement
from xml.dom import minidom

# Assume you already have your 'station_df' filtered for SiteID 501

# Create 'xml_outputs' directory if it doesn't exist
xml_output_dir = 'xml_outputs'
os.makedirs(xml_output_dir, exist_ok=True)

# Root element for the station
station_elem = Element("Station", attrib={
    "id": "501",
    "location": station_df['Location'].iloc[0],
    "geo": station_df['geo_point_2d'].iloc[0]
})

# Add measurement entries
for _, row in station_df.iterrows():
    measurement = SubElement(station_elem, "Measurement", time=str(row['Date Time']))
    for tag in ["NOx", "NO2", "NO", "PM10", "NVPM10", "VPM10", "PM2.5", "NVPM2.5",
                "VPM2.5", "CO", "O3", "SO2", "Temperature", "RH", "Air Pressure"]:
        child = SubElement(measurement, tag.replace(" ", "").replace(".", ""))
        child.text = str(row[tag]) if pd.notnull(row[tag]) else ""

# Pretty-print the XML
xml_str = minidom.parseString(ET.tostring(station_elem)).toprettyxml(indent="  ")

# Save the XML to a file
xml_output_path = os.path.join(xml_output_dir, 'station_501.xml')
with open(xml_output_path, 'w', encoding='utf-8') as f:
    f.write(xml_str)

print(f"XML file successfully saved at {xml_output_path}")


XML file successfully saved at xml_outputs\station_501.xml


# REFLECTIVE REPORT/ OVERVIEW

This project involves processing, cleaning, and inserting air quality measurement data into a structured MySQL database. The goal is to create an efficient pipeline for handling environmental data, enabling future analysis and visualization.
The dataset used was originally sourced from a cleaned CSV file containing air pollution metrics across various UK monitoring sites.

The project started off with cleaning and cropping the air pollution data set by deleting any records before 00:00 1 Jan 2010, and filtering for and removing any dud records where there were no values for SiteID or there were mismatches between SiteID and Location. 


While trying to create and populate the database, I went through a couple of code trials trying to figure out how to loop through the file while populating the database. For this, I used Xampp and a free version of phpmyadmin to host an SQL database. Then, I used pymysql, a Python package, to connect the Python environment to SQL and to write and execute SQL query languages with Python. Also, I used a sample of the data (10% of the original data) because the original data would take too long to execute.

When trying to clean the dataset and remove the null values in the file. It took me some time to figure it out because pandas made the empty fields Nan. This made it difficult to populate the database because i kept getting an error that sql does not accept nan values. So, I created a numpy array first to alter the nan values to 'none', an expression that was finally recognized by SQL.

Then, I also created NoSQL databases in two formats (key-value format and XML document format) for a select sample row of data.
